In [0]:
# =========================================================
# SILVER LAYER ETL — WITH ERROR HANDLING + LOGGING
# =========================================================
# Every table read, transform, write and validation step
# is wrapped in run_step() which:
#   - catches any exception
#   - logs it to traffic_catalog.logs.error_log
#   - marks critical steps (stops pipeline on failure)
#   - marks non-critical steps (logs and continues)
# =========================================================

import builtins
from functools import reduce
from operator import add
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── Inline error handler (no separate file import needed) ─

import uuid
import traceback

ERROR_LOG_SCHEMA = StructType([
    StructField("log_id",          StringType(),    False),
    StructField("pipeline_run_id", StringType(),    False),
    StructField("layer",           StringType(),    False),
    StructField("table_name",      StringType(),    False),
    StructField("step",            StringType(),    False),
    StructField("error_type",      StringType(),    True),
    StructField("error_message",   StringType(),    True),
    StructField("stack_trace",     StringType(),    True),
    StructField("rows_expected",   LongType(),      True),
    StructField("rows_written",    LongType(),      True),
    StructField("is_critical",     BooleanType(),   False),
    StructField("logged_at",       TimestampType(), False),
])

RUN_LOG_SCHEMA = StructType([
    StructField("run_id",        StringType(),    False),
    StructField("layer",         StringType(),    False),
    StructField("started_at",    TimestampType(), False),
    StructField("ended_at",      TimestampType(), True),
    StructField("status",        StringType(),    False),
    StructField("tables_ok",     IntegerType(),   True),
    StructField("tables_failed", IntegerType(),   True),
    StructField("notes",         StringType(),    True),
])

class PipelineLogger:
    def __init__(self, spark, layer):
        self.spark         = spark
        self.layer         = layer.upper()
        self.run_id        = builtins.str(uuid.uuid4())
        self.tables_ok     = 0
        self.tables_failed = 0
        self.started_at    = datetime.now()

        spark.sql("CREATE SCHEMA IF NOT EXISTS traffic_catalog.logs")
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.error_log (
                log_id STRING, pipeline_run_id STRING, layer STRING,
                table_name STRING, step STRING, error_type STRING,
                error_message STRING, stack_trace STRING,
                rows_expected LONG, rows_written LONG,
                is_critical BOOLEAN, logged_at TIMESTAMP
            ) USING DELTA
        """)
        spark.sql("""
            CREATE TABLE IF NOT EXISTS traffic_catalog.logs.pipeline_run_log (
                run_id STRING, layer STRING, started_at TIMESTAMP,
                ended_at TIMESTAMP, status STRING,
                tables_ok INT, tables_failed INT, notes STRING
            ) USING DELTA
        """)
        self._write_run("RUNNING", "Pipeline started")
        print(f"[{self.layer}] run_id: {self.run_id} | started: {self.started_at}")

    def log_error(self, table, step, error, is_critical=False,
                  rows_expected=None, rows_written=None):
        row = {
            "log_id":          builtins.str(uuid.uuid4()),
            "pipeline_run_id": self.run_id,
            "layer":           self.layer,
            "table_name":      table,
            "step":            step.upper(),
            "error_type":      type(error).__name__,
            "error_message":   builtins.str(error)[:2000],
            "stack_trace":     traceback.format_exc()[:4000],
            "rows_expected":   rows_expected,
            "rows_written":    rows_written,
            "is_critical":     is_critical,
            "logged_at":       datetime.now(),
        }
        self.spark.createDataFrame([row], ERROR_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.error_log")
        lvl = "CRITICAL" if is_critical else "WARNING"
        print(f"  [{lvl}] {table}.{step} — {type(error).__name__}: {builtins.str(error)[:150]}")

    def log_success(self, table, rows):
        self.tables_ok += 1
        print(f"  [OK] {table} — {rows:,} rows")

    def log_failure(self, table):
        self.tables_failed += 1

    def finish(self):
        status = "FAILED" if self.tables_failed > 0 else "SUCCESS"
        self._write_run(status, f"{self.tables_ok} ok / {self.tables_failed} failed")
        print(f"\n[{self.layer}] {status} | ok: {self.tables_ok} | failed: {self.tables_failed}")
        return status

    def _write_run(self, status, notes=None):
        row = {
            "run_id": self.run_id, "layer": self.layer,
            "started_at": self.started_at,
            "ended_at": datetime.now() if status != "RUNNING" else None,
            "status": status, "tables_ok": self.tables_ok,
            "tables_failed": self.tables_failed, "notes": notes,
        }
        self.spark.createDataFrame([row], RUN_LOG_SCHEMA) \
            .write.format("delta").mode("append") \
            .saveAsTable("traffic_catalog.logs.pipeline_run_log")

def run_step(logger, table, step, fn, is_critical=True):
    """Wrap any callable in try/except. Log on failure. Raise if critical."""
    try:
        return fn()
    except Exception as e:
        logger.log_error(table, step, e, is_critical=is_critical)
        logger.log_failure(table)
        if is_critical:
            logger.finish()
            raise RuntimeError(
                f"Critical failure: {table}.{step} — run_id={logger.run_id}"
            ) from e
        return None

# =========================================================
# SILVER ETL HELPERS
# =========================================================

def safe_timestamp(col_name):
    return F.expr(f"try_to_timestamp({col_name})")

def dedup_by_event_id(df):
    w = Window.partitionBy("event_id").orderBy(F.col("event_timestamp").desc())
    return df.withColumn("rn", F.row_number().over(w)) \
             .filter(F.col("rn") == 1).drop("rn")

def add_time_of_day(df):
    return df.withColumn("time_of_day",
        F.when((F.hour(F.col("event_timestamp")) >= 6)  &
               (F.hour(F.col("event_timestamp")) < 10), F.lit("MORNING_PEAK"))
         .when((F.hour(F.col("event_timestamp")) >= 10) &
               (F.hour(F.col("event_timestamp")) < 16), F.lit("MIDDAY"))
         .when((F.hour(F.col("event_timestamp")) >= 16) &
               (F.hour(F.col("event_timestamp")) < 20), F.lit("EVENING_PEAK"))
         .when((F.hour(F.col("event_timestamp")) >= 20) &
               (F.hour(F.col("event_timestamp")) < 24), F.lit("NIGHT"))
         .otherwise(F.lit("LATE_NIGHT"))
    )

def write_silver(df, table_name, logger):
    """Write to silver with error handling and row count logging."""
    def _write():
        df.write.format("delta") \
            .mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(f"traffic_catalog.silver.{table_name}")
        rows = spark.table(f"traffic_catalog.silver.{table_name}").count()
        logger.log_success(table_name, rows)
        return rows
    return run_step(logger, table_name, "WRITE", _write, is_critical=True)

def validate_silver_table(table_name, logger):
    """Run DQ checks on a silver table and log any issues."""
    def _validate():
        df           = spark.table(f"traffic_catalog.silver.{table_name}")
        null_sensors = df.filter(F.col("sensor_id").isNull()).count()
        null_ts      = df.filter(F.col("event_timestamp").isNull()).count()
        row_count    = df.count()

        issues = []
        if null_sensors > 0:
            issues.append(f"sensor_id has {null_sensors} nulls")
        if null_ts > 0:
            issues.append(f"event_timestamp has {null_ts} nulls")
        if row_count == 0:
            issues.append("table is empty")

        if issues:
            raise ValueError(f"DQ failed: {'; '.join(issues)}")

        print(f"  [DQ] {table_name} — {row_count:,} rows | sensor_id nulls: {null_sensors} | ts nulls: {null_ts}")
        return row_count

    # DQ failures are warnings — log but don't stop pipeline
    return run_step(logger, table_name, "VALIDATE", _validate, is_critical=False)

# =========================================================
# START PIPELINE
# =========================================================

logger = PipelineLogger(spark, layer="SILVER")

print("\n── INCIDENT TABLE ──")

incident_df = run_step(logger, "incident_clean", "READ",
    lambda: spark.table("traffic_catalog.bronze.incident").select(
        F.regexp_replace(F.col("event_id"), "[^a-zA-Z0-9_]", "").alias("event_id"),
        F.col("sensor_id").cast(StringType()),
        safe_timestamp("event_timestamp").alias("event_timestamp"),
        F.col("incident_id").cast(StringType()),
        F.upper(F.coalesce(F.trim(F.col("incident_type")),     F.lit("UNKNOWN"))).alias("incident_type"),
        F.upper(F.coalesce(F.trim(F.col("incident_severity")), F.lit("UNKNOWN"))).alias("incident_severity"),
        F.coalesce(F.col("incident_duration").cast(DoubleType()),  F.lit(0.0)).alias("incident_duration"),
        F.coalesce(F.col("vehicles_affected").cast(IntegerType()), F.lit(0)).alias("vehicles_affected"),
        F.upper(F.coalesce(F.trim(F.col("lane_blocked")), F.lit("UNKNOWN"))).alias("lane_blocked"),
        F.coalesce(F.col("response_time").cast(DoubleType()), F.lit(0.0)).alias("response_time")
    ), is_critical=True)

if incident_df is not None:
    incident_df = run_step(logger, "incident_clean", "TRANSFORM", lambda: (
        dedup_by_event_id(
            incident_df
            .filter(F.col("event_id").isNotNull())
            .filter(F.col("sensor_id").isNotNull())
            .filter(F.col("incident_id").isNotNull())
        )
        .filter((F.col("incident_duration") >= 0) &
                (F.col("response_time")     >= 0) &
                (F.col("vehicles_affected") >= 0))
        .withColumn("lane_blocked_flag",
            F.when(F.col("lane_blocked") == "YES",  F.lit(True))
             .when(F.col("lane_blocked") == "NO",   F.lit(False))
             .otherwise(F.lit(None).cast(BooleanType())))
        .withColumn("is_severe", F.col("incident_severity") == "HIGH")
        .transform(add_time_of_day)
        .withColumn("is_peak_hour",
            F.col("time_of_day").isin("MORNING_PEAK", "EVENING_PEAK"))
    ), is_critical=True)

    if incident_df is not None:
        write_silver(incident_df, "incident_clean", logger)

print("\n── SENSOR TABLE ──")

sensor_df = run_step(logger, "sensor_clean", "READ",
    lambda: spark.table("traffic_catalog.bronze.sensor").select(
        F.regexp_replace(F.col("event_id"), "[^a-zA-Z0-9_]", "").alias("event_id"),
        F.col("sensor_id").cast(StringType()),
        safe_timestamp("event_timestamp").alias("event_timestamp"),
        F.col("location_id").cast(StringType()),
        F.col("lane_id").cast(StringType()),
        F.coalesce(F.col("vehicle_count").cast(IntegerType()),   F.lit(0)).alias("vehicle_count"),
        F.coalesce(F.col("avg_speed").cast(DoubleType()),        F.lit(0.0)).alias("avg_speed"),
        F.coalesce(F.col("traffic_density").cast(DoubleType()),  F.lit(0.0)).alias("traffic_density"),
        F.coalesce(F.col("occupancy_rate").cast(DoubleType()),   F.lit(0.0)).alias("occupancy_rate"),
        F.coalesce(F.col("congestion_score").cast(DoubleType()), F.lit(0.0)).alias("congestion_score")
    ), is_critical=True)

if sensor_df is not None:
    sensor_df = run_step(logger, "sensor_clean", "TRANSFORM", lambda: (
        dedup_by_event_id(
            sensor_df
            .filter(F.col("event_id").isNotNull())
            .filter(F.col("sensor_id").isNotNull())
        )
        .filter((F.col("vehicle_count")   >= 0) &
                (F.col("avg_speed")       >= 0) & (F.col("avg_speed") <= 200) &
                (F.col("traffic_density") >= 0) &
                F.col("occupancy_rate").between(0, 1) &
                F.col("congestion_score").between(0, 100))
        .withColumn("congestion_level",
            F.when(F.col("congestion_score") <  25, F.lit("LOW"))
             .when(F.col("congestion_score") <  50, F.lit("MEDIUM"))
             .when(F.col("congestion_score") <  75, F.lit("HIGH"))
             .otherwise(F.lit("CRITICAL")))
        .transform(add_time_of_day)
        .withColumn("is_peak_hour",
            F.col("time_of_day").isin("MORNING_PEAK", "EVENING_PEAK"))
    ), is_critical=True)

    if sensor_df is not None:
        write_silver(sensor_df, "sensor_clean", logger)

print("\n── SPEED TABLE ──")

speed_df = run_step(logger, "speed_clean", "READ",
    lambda: spark.table("traffic_catalog.bronze.speed").select(
        F.regexp_replace(F.col("event_id"), "[^a-zA-Z0-9_]", "").alias("event_id"),
        F.col("sensor_id").cast(StringType()),
        safe_timestamp("event_timestamp").alias("event_timestamp"),
        F.coalesce(F.col("speed_limit").cast(IntegerType()),           F.lit(0)).alias("speed_limit"),
        F.coalesce(F.col("avg_speed").cast(DoubleType()),              F.lit(0.0)).alias("avg_speed"),
        F.coalesce(F.col("min_speed").cast(DoubleType()),              F.lit(0.0)).alias("min_speed"),
        F.coalesce(F.col("max_speed").cast(DoubleType()),              F.lit(0.0)).alias("max_speed"),
        F.coalesce(F.col("speed_variance").cast(DoubleType()),         F.lit(0.0)).alias("speed_variance"),
        F.coalesce(F.col("speed_violation_count").cast(IntegerType()), F.lit(0)).alias("speed_violation_count"),
        F.coalesce(F.col("speed_index").cast(DoubleType()),            F.lit(0.0)).alias("speed_index")
    ), is_critical=True)

if speed_df is not None:
    speed_df = run_step(logger, "speed_clean", "TRANSFORM", lambda: (
        dedup_by_event_id(
            speed_df
            .filter(F.col("event_id").isNotNull())
            .filter(F.col("sensor_id").isNotNull())
        )
        .filter((F.col("speed_limit")           >  0) &
                (F.col("avg_speed")             >= 0) & (F.col("avg_speed") <= 200) &
                (F.col("min_speed")             >= 0) &
                (F.col("max_speed")             >= 0) & (F.col("max_speed") <= 200) &
                (F.col("speed_violation_count") >= 0) &
                (F.col("speed_index")           >= 0))
        .filter((F.col("min_speed") <= F.col("avg_speed")) &
                (F.col("avg_speed") <= F.col("max_speed")))
        .withColumn("speed_violation_flag",
            F.when(F.col("avg_speed") > F.col("speed_limit"), F.lit(True))
             .otherwise(F.lit(False)))
        .withColumn("speed_excess",
            F.round(F.when(F.col("avg_speed") > F.col("speed_limit"),
                           F.col("avg_speed") - F.col("speed_limit"))
                     .otherwise(F.lit(0.0)), 2))
        .transform(add_time_of_day)
        .withColumn("is_peak_hour",
            F.col("time_of_day").isin("MORNING_PEAK", "EVENING_PEAK"))
    ), is_critical=True)

    if speed_df is not None:
        write_silver(speed_df, "speed_clean", logger)

print("\n── VEHICLE TABLE ──")

vehicle_df = run_step(logger, "vehicle_clean", "READ",
    lambda: spark.table("traffic_catalog.bronze.vehicle").select(
        F.regexp_replace(F.col("event_id"), "[^a-zA-Z0-9_]", "").alias("event_id"),
        F.col("sensor_id").cast(StringType()),
        safe_timestamp("event_timestamp").alias("event_timestamp"),
        F.upper(F.coalesce(F.trim(F.col("vehicle_type")), F.lit("UNKNOWN"))).alias("vehicle_type"),
        F.coalesce(F.col("vehicle_count").cast(IntegerType()),    F.lit(0)).alias("vehicle_count"),
        F.coalesce(F.col("vehicle_flow_rate").cast(DoubleType()), F.lit(0.0)).alias("vehicle_flow_rate"),
        F.coalesce(F.col("vehicle_density").cast(DoubleType()),   F.lit(0.0)).alias("vehicle_density"),
        F.coalesce(F.col("queue_length").cast(DoubleType()),      F.lit(0.0)).alias("queue_length"),
        F.coalesce(F.col("lane_utilization").cast(DoubleType()),  F.lit(0.0)).alias("lane_utilization"),
        F.coalesce(F.col("vehicle_delay").cast(DoubleType()),     F.lit(0.0)).alias("vehicle_delay")
    ), is_critical=True)

if vehicle_df is not None:
    vehicle_df = run_step(logger, "vehicle_clean", "TRANSFORM", lambda: (
        dedup_by_event_id(
            vehicle_df
            .filter(F.col("event_id").isNotNull())
            .filter(F.col("sensor_id").isNotNull())
        )
        .filter((F.col("vehicle_count")     >= 0) &
                (F.col("vehicle_flow_rate") >= 0) &
                (F.col("vehicle_density")   >= 0) &
                (F.col("queue_length")      >= 0) &
                F.col("lane_utilization").between(0, 1) &
                (F.col("vehicle_delay")     >= 0))
        .withColumn("is_heavy_vehicle",
            F.col("vehicle_type").isin("TRUCK", "BUS"))
        .transform(add_time_of_day)
        .withColumn("is_peak_hour",
            F.col("time_of_day").isin("MORNING_PEAK", "EVENING_PEAK"))
    ), is_critical=True)

    if vehicle_df is not None:
        write_silver(vehicle_df, "vehicle_clean", logger)

print("\n── SIGNAL TABLE ──")

signal_df = run_step(logger, "signal_clean", "READ",
    lambda: spark.table("traffic_catalog.bronze.signal").select(
        F.regexp_replace(F.col("event_id"), "[^a-zA-Z0-9_]", "").alias("event_id"),
        F.col("sensor_id").cast(StringType()),
        safe_timestamp("event_timestamp").alias("event_timestamp"),
        F.col("signal_id").cast(StringType()),
        F.coalesce(F.col("signal_cycle_time").cast(DoubleType()), F.lit(0.0)).alias("signal_cycle_time"),
        F.coalesce(F.col("green_time").cast(DoubleType()),        F.lit(0.0)).alias("green_time"),
        F.coalesce(F.col("red_time").cast(DoubleType()),          F.lit(0.0)).alias("red_time"),
        F.coalesce(F.col("signal_wait_time").cast(DoubleType()),  F.lit(0.0)).alias("signal_wait_time"),
        F.coalesce(F.col("queue_length").cast(DoubleType()),      F.lit(0.0)).alias("queue_length"),
        F.coalesce(F.col("signal_efficiency").cast(DoubleType()), F.lit(0.0)).alias("signal_efficiency")
    ), is_critical=True)

if signal_df is not None:
    signal_df = run_step(logger, "signal_clean", "TRANSFORM", lambda: (
        dedup_by_event_id(
            signal_df
            .filter(F.col("event_id").isNotNull())
            .filter(F.col("sensor_id").isNotNull())
            .filter(F.col("signal_id").isNotNull())
        )
        .filter((F.col("signal_cycle_time") >  0) &
                (F.col("green_time")        >= 0) &
                (F.col("red_time")          >= 0) &
                (F.col("signal_wait_time")  >= 0) &
                (F.col("queue_length")      >= 0) &
                F.col("signal_efficiency").between(0, 1))
        .filter((F.col("green_time") + F.col("red_time")) <= F.col("signal_cycle_time"))
        .withColumn("green_ratio",
            F.round(F.col("green_time") / F.col("signal_cycle_time"), 4))
        .withColumn("red_ratio",
            F.round(F.col("red_time") / F.col("signal_cycle_time"), 4))
        .withColumn("signal_status",
            F.when(F.col("signal_efficiency") >= 0.75, F.lit("GOOD"))
             .when(F.col("signal_efficiency") >= 0.50, F.lit("MODERATE"))
             .otherwise(F.lit("POOR")))
        .transform(add_time_of_day)
        .withColumn("is_peak_hour",
            F.col("time_of_day").isin("MORNING_PEAK", "EVENING_PEAK"))
    ), is_critical=True)

    if signal_df is not None:
        write_silver(signal_df, "signal_clean", logger)

# =========================================================
# TIMESTAMP CLEANING
# =========================================================

print("\n── TIMESTAMP CLEANING ──")

tables = ["incident_clean", "sensor_clean", "speed_clean",
          "vehicle_clean",  "signal_clean"]

for table in tables:
    def _ts_clean(t=table):
        df = spark.table(f"traffic_catalog.silver.{t}")
        if "event_timestamp" not in df.columns:
            return

        fw = Window.partitionBy("sensor_id").orderBy("event_timestamp") \
                   .rowsBetween(Window.unboundedPreceding, 0)
        bw = Window.partitionBy("sensor_id").orderBy("event_timestamp") \
                   .rowsBetween(0, Window.unboundedFollowing)

        df = df.withColumn("event_timestamp",
                F.last(F.col("event_timestamp"), True).over(fw)) \
               .withColumn("event_timestamp",
                F.first(F.col("event_timestamp"), True).over(bw)) \
               .withColumn("event_timestamp",
                F.coalesce(F.col("event_timestamp"), F.current_timestamp()))

        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"traffic_catalog.silver.{t}")

    run_step(logger, table, "TIMESTAMP_CLEAN", _ts_clean, is_critical=False)

print("Timestamp cleaning complete")

# =========================================================
# TIMESTAMP SPLITTING
# =========================================================

print("\n── TIMESTAMP SPLITTING ──")

for table in tables:
    def _ts_split(t=table):
        df = spark.table(f"traffic_catalog.silver.{t}")
        df = df \
            .withColumn("event_year",    F.year(F.col("event_timestamp"))) \
            .withColumn("event_month",   F.month(F.col("event_timestamp"))) \
            .withColumn("event_day",     F.dayofmonth(F.col("event_timestamp"))) \
            .withColumn("event_hour",    F.hour(F.col("event_timestamp"))) \
            .withColumn("event_minute",  F.minute(F.col("event_timestamp"))) \
            .withColumn("event_date",    F.to_date(F.col("event_timestamp"))) \
            .withColumn("event_weekday", F.date_format(F.col("event_timestamp"), "EEEE")) \
            .withColumn("event_week",    F.weekofyear(F.col("event_timestamp"))) \
            .withColumn("time_of_day",
                F.when((F.col("event_hour") >= 6)  & (F.col("event_hour") < 10), F.lit("MORNING_PEAK"))
                 .when((F.col("event_hour") >= 10) & (F.col("event_hour") < 16), F.lit("MIDDAY"))
                 .when((F.col("event_hour") >= 16) & (F.col("event_hour") < 20), F.lit("EVENING_PEAK"))
                 .when((F.col("event_hour") >= 20) & (F.col("event_hour") < 24), F.lit("NIGHT"))
                 .otherwise(F.lit("LATE_NIGHT")))
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"traffic_catalog.silver.{t}")

    run_step(logger, table, "TIMESTAMP_SPLIT", _ts_split, is_critical=False)

print("Timestamp splitting complete")

# =========================================================
# OUTLIER FLAGGING
# =========================================================

print("\n── OUTLIER FLAGGING ──")

outlier_configs = {
    "sensor_clean":   ["avg_speed", "traffic_density", "congestion_score", "vehicle_count"],
    "speed_clean":    ["avg_speed", "speed_variance", "speed_violation_count"],
    "vehicle_clean":  ["vehicle_count", "vehicle_flow_rate", "queue_length", "vehicle_delay"],
    "signal_clean":   ["signal_wait_time", "queue_length", "signal_efficiency"],
    "incident_clean": ["incident_duration", "response_time", "vehicles_affected"]
}

for table, col_names in outlier_configs.items():
    def _outlier(t=table, cols=col_names):
        df = spark.table(f"traffic_catalog.silver.{t}")
        agg_exprs = [e for c in cols
                     for e in [F.mean(F.col(c)).alias(f"{c}_mean"),
                                F.stddev(F.col(c)).alias(f"{c}_std")]]
        stats = df.select(agg_exprs).collect()[0]
        outlier_expr = F.lit(False)
        for c in cols:
            m, s = stats[f"{c}_mean"], stats[f"{c}_std"]
            if m is not None and s is not None and s > 0:
                outlier_expr = outlier_expr | (F.abs(F.col(c) - m) > (3 * s))
        df = df.withColumn("is_outlier", outlier_expr)
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"traffic_catalog.silver.{t}")
        flagged = df.filter(F.col("is_outlier") == True).count()
        print(f"  {t} — {flagged} outlier rows flagged")

    run_step(logger, table, "OUTLIER_FLAG", _outlier, is_critical=False)

# =========================================================
# DQ SCORING
# =========================================================

print("\n── DQ SCORING ──")

dq_configs = {
    "incident_clean": ["incident_duration", "response_time", "vehicles_affected"],
    "sensor_clean":   ["vehicle_count", "avg_speed", "traffic_density",
                       "occupancy_rate", "congestion_score"],
    "speed_clean":    ["avg_speed", "min_speed", "max_speed",
                       "speed_variance", "speed_violation_count", "speed_index"],
    "vehicle_clean":  ["vehicle_count", "vehicle_flow_rate", "vehicle_density",
                       "queue_length", "lane_utilization", "vehicle_delay"],
    "signal_clean":   ["signal_cycle_time", "green_time", "red_time",
                       "signal_wait_time", "queue_length", "signal_efficiency"]
}

for table, col_list in dq_configs.items():
    def _dq(t=table, cols=col_list):
        df = spark.table(f"traffic_catalog.silver.{t}")
        dq_expr = reduce(add,
            [F.when(F.col(c) == 0, F.lit(1)).otherwise(F.lit(0)) for c in cols])
        df = df.withColumn("dq_score", dq_expr.cast(IntegerType()))
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(f"traffic_catalog.silver.{t}")
        saved  = spark.table(f"traffic_catalog.silver.{t}")
        total  = int(saved.count())
        clean  = int(saved.filter(F.col("dq_score") == 0).count())
        pct    = builtins.round(clean / total * 100, 1)
        print(f"  {t} — {clean}/{total} fully populated ({pct}%)")

    run_step(logger, table, "DQ_SCORE", _dq, is_critical=False)

# =========================================================
# PARTITIONED WRITE
# =========================================================

print("\n── PARTITIONED WRITE ──")

for table in tables:
    def _partition(t=table):
        df = spark.table(f"traffic_catalog.silver.{t}")
        df.write.format("delta").mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("event_year", "event_month") \
            .saveAsTable(f"traffic_catalog.silver.{t}")
        print(f"  {t} — partitioned write complete")

    run_step(logger, table, "PARTITION_WRITE", _partition, is_critical=False)

# =========================================================
# FINAL DQ VALIDATION
# =========================================================

print("\n── SILVER DQ VALIDATION ──")

for table in tables:
    validate_silver_table(table, logger)

# =========================================================
# FINISH
# =========================================================

final_status = logger.finish()

print("\n==============================")
print("SILVER PIPELINE COMPLETE")
print("==============================")
print(f"Status   : {final_status}")
print(f"Run ID   : {logger.run_id}")
print(f"Check logs: SELECT * FROM traffic_catalog.logs.error_log "
      f"WHERE pipeline_run_id = '{logger.run_id}'")